# E6 — Optimizer De-confound: architecture vs. optimizer effect on fn*

**Purpose.** In the paper, the optimizer is tied to the architecture (Adam→MLP-5,
SGD→ResNet-20), so the headline architecture effect on the feature-norm threshold
fn* (+458% on MNIST: MLP-5 1.052 → ResNet-20 5.867) is confounded with the
optimizer. This notebook breaks the confound by training **both** architectures on
MNIST under **both** optimizers, using identical optimizer settings across
architectures, and reports a 2×2 fn* table.

**Reads the result as follows.** If the large MLP→ResNet gap in fn* persists under a
*common* optimizer (e.g. ResNet-20/MNIST under Adam still ≫ MLP-5/MNIST under Adam),
the effect is driven by architecture, not optimizer — which de-confounds the
paper's claim. If the gap shrinks substantially when the optimizer is held fixed,
the claim must be attributed jointly to architecture and optimizer.

**Protocol is identical to the published runs** (model defs, two-phase CE→MSE,
`compute_nc`, NC1<0.01 strict / <0.05 relaxed, 3 seeds, no augmentation). The only
varied factor is the optimizer. Per-seed CSVs use the same schema as the released
logs (`epoch,phase,train,test,nc1,nc2,nc3,feat_norm`).

**Known reference values** (published): MLP-5/MNIST (Adam) fn*=1.052 (N=12);
ResNet-20/MNIST (SGD) fn*=5.867 (N=3). The two reproduction cells below should land
near these; the two new cells (ResNet-20/Adam, MLP-5/SGD) are the de-confound.

> Run on an A100 (Colab) or GPU (Kaggle). MNIST is fast; expect well under an hour.
> Set `RUN_REPRODUCTIONS=False` to run only the two missing cells.


In [ ]:
import torch, torchvision, time, os, sys
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Output dir auto-detected (Kaggle / Colab / local), mirroring the other notebooks.
if os.path.isdir('/kaggle/working'):
    PLATFORM, SAVE_DIR, DATA_DIR = 'kaggle', '/kaggle/working/', '/kaggle/working/data/'
elif 'google.colab' in sys.modules or os.path.isdir('/content'):
    PLATFORM = 'colab'
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = '/content/drive/MyDrive/nc_outputs/'
        print('Drive mounted ->', SAVE_DIR)
    except Exception as exc:
        print('WARNING: Drive mount failed (%s). Using /content/ (not crash-safe).' % exc)
        SAVE_DIR = '/content/'
    DATA_DIR = '/content/data/'
else:
    PLATFORM, SAVE_DIR, DATA_DIR = 'local', './', './data/'
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
assert torch.cuda.is_available(), 'No GPU - enable an A100 (Colab) or GPU (Kaggle).'
print('Platform:', PLATFORM, '| SAVE_DIR:', SAVE_DIR)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

Mounted at /content/drive
Drive mounted -> /content/drive/MyDrive/nc_outputs/
Platform: colab | SAVE_DIR: /content/drive/MyDrive/nc_outputs/
GPU: NVIDIA A100-SXM4-80GB | Torch: 2.11.0+cu128


In [ ]:
# ---- MNIST data, two views ----------------------------------------------------
# MLP view: 28x28 grayscale, flattened to 784 inside the model.
mlp_tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
# ResNet view: pad to 32x32, repeat to 3 channels (matches the published
# ResNet-20/MNIST cell exactly). No augmentation (standard for NC).
resnet_tf = T.Compose([
    T.Pad(2), T.ToTensor(), T.Normalize((0.1307,), (0.3081,)),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),
])

def make_loaders(transform, batch_train=512, batch_test=1024):
    tr = torchvision.datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
    te = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
    return (DataLoader(tr, batch_train, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(te, batch_test,  shuffle=False, num_workers=2, pin_memory=True))

mlp_train,    mlp_test    = make_loaders(mlp_tf)
resnet_train, resnet_test = make_loaders(resnet_tf)
print('MNIST loaders ready (MLP 784 view + ResNet 32x32x3 view).')

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 342kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.12MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.71MB/s]

MNIST loaders ready (MLP 784 view + ResNet 32x32x3 view).


In [ ]:
# ---- Models (identical to the published notebooks) ---------------------------
class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10, in_dim=784):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(in_dim, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu'); nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                                      nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.skip(x))

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.layer1 = self._make(16, 16, 3, 1)
        self.layer2 = self._make(16, 32, 3, 2)
        self.layer3 = self._make(32, 64, 3, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):   nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def _make(self, in_c, out_c, n, stride):
        layers = [BasicBlock(in_c, out_c, stride)] + [BasicBlock(out_c, out_c, 1) for _ in range(n-1)]
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(self.pool(x).flatten(1))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.fc.weight.detach()
print('Models defined.')

Models defined.


In [ ]:
# ---- NC metrics + accuracy (identical to the published notebooks) ------------
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval(); fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)).cpu()); ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0); mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M = mu_c - mu_G
    Sw = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c]) for c in range(K)) / len(H)
    Sb = M.T @ M / K
    nc1 = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    Mn = F.normalize(M, dim=1); cos = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2 = (cos[mask] - (-1./(K-1))).abs().mean().item()
    Wn = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3 = (1 - (Mn * Wn).sum(1).mean()).item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3, 'feat_norm': H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item(); total += len(y)
    return correct / total
print('Metrics ready.')

Metrics ready.


In [ ]:
# ---- Unified two-phase runner; optimizer is the ONLY varied factor -----------
def make_opt_sched(params, optimizer_type, n_ep, phase):
    if optimizer_type == 'adam':                       # MLP-style (paper)
        opt = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
    elif optimizer_type == 'sgd':                      # ResNet-style (paper)
        opt = torch.optim.SGD(params, lr=0.1, momentum=0.9, weight_decay=1e-3, nesterov=True)
        milestones = [100, 150] if phase == 1 else [300, 450]
        sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=milestones, gamma=0.1)
    else:
        raise ValueError(optimizer_type)
    return opt, sch

def run_twophase(model, train_loader, test_loader, name, optimizer_type,
                 phase1=200, phase2=600, nc_every=10, K=10):
    model = model.to(DEVICE)
    rows = []; terminal = False
    t_nc_strict = fn_strict = t_nc_relaxed = fn_relaxed = None
    t0 = time.time()
    for phase, loss_fn, n_ep in [(1, 'ce', phase1), (2, 'mse', phase2)]:
        opt, sch = make_opt_sched(model.parameters(), optimizer_type, n_ep, phase)
        off = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l; model.train()
            for x, y in train_loader:
                x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = F.mse_loss(logits, F.one_hot(y, K).float()) if loss_fn == 'mse' \
                       else F.cross_entropy(logits, y)
                if not torch.isfinite(loss):
                    print('  [%s] non-finite loss at ep %d -> aborting this run' % (name, ep))
                    return pd.DataFrame(rows), t_nc_strict, fn_strict, t_nc_relaxed, fn_relaxed
                loss.backward(); opt.step()
            sch.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader); te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True; print('  [%s] terminal phase at ep %d' % (name, ep))
                nc = compute_nc(model, train_loader, K) if (terminal or phase == 2) \
                     else {'nc1': None,'nc2': None,'nc3': None,'feat_norm': None}
                rows.append({'epoch': ep, 'phase': phase, 'train': tr, 'test': te, **nc})
                if nc['nc1'] is not None:
                    if t_nc_relaxed is None and nc['nc1'] < 0.05:
                        t_nc_relaxed, fn_relaxed = ep, nc['feat_norm']
                    if t_nc_strict is None and nc['nc1'] < 0.01:
                        t_nc_strict, fn_strict = ep, nc['feat_norm']
                        print('  [%s] NC1<0.01 at ep %d  fn=%.4f' % (name, ep, fn_strict))
    print('  [%s] done in %.1f min' % (name, (time.time()-t0)/60))
    return pd.DataFrame(rows), t_nc_strict, fn_strict, t_nc_relaxed, fn_relaxed
print('run_twophase ready.')

run_twophase ready.


In [ ]:
# ---- Run the 2x2 (architecture x optimizer) grid on MNIST --------------------
RUN_REPRODUCTIONS = True   # set False to run only the two NEW cells (resnet/adam, mlp/sgd)
SEEDS = range(3)

# (arch, optimizer, is_new): the two NEW cells are the de-confound; the other two
# reproduce the published values under one notebook/environment for a clean table.
CONFIG = [
    ('resnet', 'adam', True),   # NEW: ResNet-20/MNIST under Adam   (vs MLP/Adam 1.052)
    ('mlp',    'sgd',  True),   # NEW: MLP-5/MNIST under SGD         (vs ResNet/SGD 5.867)
    ('mlp',    'adam', False),  # reproduce MLP-5/MNIST  (~1.052)
    ('resnet', 'sgd',  False),  # reproduce ResNet-20/MNIST (~5.867)
]

def build(arch):
    if arch == 'mlp':    return MLP(depth=5, width=512, act_cls=nn.ReLU), mlp_train, mlp_test
    if arch == 'resnet': return ResNet20(), resnet_train, resnet_test

rows = []
for arch, opt_type, is_new in CONFIG:
    if not is_new and not RUN_REPRODUCTIONS:
        continue
    for seed in SEEDS:
        tag = '%s_mnist_%s_s%d' % (arch, opt_type, seed)
        print('\n=== %s (%s) ===' % (tag, 'NEW' if is_new else 'reproduction'))
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
        model, tl, vl = build(arch)
        df, ts, fs, tr_, fr = run_twophase(model, tl, vl, tag, opt_type)
        df.to_csv(SAVE_DIR + 'deconfound_' + tag + '.csv', index=False)
        rows.append({'arch': arch, 'optimizer': opt_type, 'seed': seed,
                     'is_new': is_new, 'T_NC_strict': ts, 'fn_strict': fs,
                     'T_NC_relaxed': tr_, 'fn_relaxed': fr,
                     'test_acc_final': df.test.iloc[-1] if len(df) else None})
        pd.DataFrame(rows).to_csv(SAVE_DIR + 'deconfound_summary.csv', index=False)

summ = pd.DataFrame(rows)
print('\n================ DE-CONFOUND SUMMARY ================')
print(summ.to_string(index=False))

def cell_fn(arch, opt_type):
    s = summ[(summ.arch==arch) & (summ.optimizer==opt_type)].dropna(subset=['fn_strict'])
    if len(s)==0:
        s = summ[(summ.arch==arch) & (summ.optimizer==opt_type)].dropna(subset=['fn_relaxed'])
        return (s.fn_relaxed.mean(), s.fn_relaxed.std(), len(s), 'NC1<0.05') if len(s) else (None,None,0,'-')
    return (s.fn_strict.mean(), s.fn_strict.std(), len(s), 'NC1<0.01')

print('\nfn* by (architecture x optimizer) on MNIST:')
print('%-12s %-18s %-18s' % ('', 'Adam', 'SGD'))
for arch in ['mlp','resnet']:
    cells_txt=[]
    for opt_type in ['adam','sgd']:
        m,sd,n,crit = cell_fn(arch,opt_type)
        cells_txt.append(('%.3f+/-%.3f (N=%d,%s)'%(m,sd,n,crit)) if m is not None else 'DNF/none')
    print('%-12s %-18s %-18s' % (arch, cells_txt[0], cells_txt[1]))

# Architecture effect under each fixed optimizer (the de-confound)
for opt_type in ['adam','sgd']:
    mm,_,_,_ = cell_fn('mlp',opt_type); rr,_,_,_ = cell_fn('resnet',opt_type)
    if mm and rr:
        print('Architecture effect (MLP->ResNet) under %s: %+.3f  (%+.0f%%)'
              % (opt_type.upper(), rr-mm, (rr-mm)/mm*100))
print('\nInterpretation: if the MLP->ResNet gap stays large under BOTH optimizers,')
print('the architecture effect on fn* is not an optimizer artifact.')


=== resnet_mnist_adam_s0 (NEW) ===
  [resnet_mnist_adam_s0] terminal phase at ep 20
  [resnet_mnist_adam_s0] NC1<0.01 at ep 150  fn=7.4248
  [resnet_mnist_adam_s0] done in 160.2 min

=== resnet_mnist_adam_s1 (NEW) ===
  [resnet_mnist_adam_s1] terminal phase at ep 10
  [resnet_mnist_adam_s1] NC1<0.01 at ep 130  fn=7.6129
  [resnet_mnist_adam_s1] done in 159.4 min

=== resnet_mnist_adam_s2 (NEW) ===
  [resnet_mnist_adam_s2] terminal phase at ep 30
  [resnet_mnist_adam_s2] NC1<0.01 at ep 160  fn=7.3669
  [resnet_mnist_adam_s2] done in 159.2 min

=== mlp_mnist_sgd_s0 (NEW) ===
  [mlp_mnist_sgd_s0] terminal phase at ep 30
  [mlp_mnist_sgd_s0] NC1<0.01 at ep 210  fn=0.0000
  [mlp_mnist_sgd_s0] done in 100.9 min

=== mlp_mnist_sgd_s1 (NEW) ===
  [mlp_mnist_sgd_s1] terminal phase at ep 40
  [mlp_mnist_sgd_s1] NC1<0.01 at ep 210  fn=0.0000
  [mlp_mnist_sgd_s1] done in 101.0 min

=== mlp_mnist_sgd_s2 (NEW) ===
  [mlp_mnist_sgd_s2] terminal phase at ep 50
  [mlp_mnist_sgd_s2] NC1<0.01 at ep 210 

In [ ]:
# ---- Export ------------------------------------------------------------------
import glob
out = sorted(glob.glob(SAVE_DIR + 'deconfound_*.csv'))
print('Outputs (%d):' % len(out)); [print(' ', f) for f in out]
if PLATFORM == 'colab' and not SAVE_DIR.startswith('/content/drive'):
    from google.colab import files
    for f in out: files.download(f)

In [ ]:
import re, numpy as np, matplotlib.pyplot as plt

TAG_RE = re.compile(r'(mlp|resnet)\w*?_(\w+?)_(adam|sgd)_s(\d+)', re.I)

def _to_fn(v):
    """Pull a scalar fn* from a stored value (float, dict, or per-epoch array)."""
    if isinstance(v, (int, float)):              return float(v)
    if isinstance(v, dict):
        for k in ('fn_star', 'fn', 'feature_norm', 'fnorm', 'fnstar'):
            if k in v: return _to_fn(v[k])
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.asarray(v, float); a = a[np.isfinite(a)]
        if a.size: return float(a[-1])           # last logged epoch
    return None

def harvest():
    """Find every (tag, fn*) pair living in the kernel's globals."""
    found = {}
    for var in list(globals().values()):
        items = (var.items() if isinstance(var, dict)
                 else enumerate(var) if isinstance(var, (list, tuple)) else [])
        for key, val in items:
            tag = key if isinstance(key, str) else None
            if tag is None and isinstance(val, dict):          # list-of-dicts
                tag = val.get('tag') or val.get('name') or val.get('run')
            if not (isinstance(tag, str) and TAG_RE.search(tag)):
                continue
            fn = _to_fn(val)
            if fn is not None:
                found[tag] = fn                                # last writer wins
    return found

runs = harvest()
print(f'harvested {len(runs)} runs:')
for t, f in sorted(runs.items()): print(f'  {t:28s} fn*={f:.4f}')
if not runs:
    raise RuntimeError("No runs found. Set:  runs = {tag: fn, ...}  from your results variable.")

# ---- bucket by (arch, optimiser) ----
buckets = {}
for tag, fn in runs.items():
    arch, _ds, opt, _seed = TAG_RE.search(tag).groups()
    buckets.setdefault((arch.lower(), opt.lower()), []).append(fn)

def stats(key):
    v = buckets.get(key, [])
    return (np.mean(v) if v else np.nan,
            np.std(v, ddof=1) if len(v) > 1 else 0.0, v)

archs = ['mlp', 'resnet']; labels = ['MLP-5', 'ResNet-20']
adam = [stats((a, 'adam')) for a in archs]
sgd  = [stats((a, 'sgd'))  for a in archs]

x = np.arange(len(archs)); w = 0.36
fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.bar(x - w/2, [m for m,_,_ in adam], w, yerr=[s for _,s,_ in adam], capsize=4,
       color='#4C72B0', edgecolor='black', linewidth=0.6, label='Adam (matched)')
ax.bar(x + w/2, [m for m,_,_ in sgd],  w, yerr=[s for _,s,_ in sgd],  capsize=4,
       color='#C44E52', edgecolor='black', linewidth=0.6, label='SGD (main grid)')

for xi, (_,_,v) in zip(x - w/2, adam): ax.scatter(np.full(len(v), xi), v, c='k', s=14, zorder=3)
for xi, (_,_,v) in zip(x + w/2, sgd):  ax.scatter(np.full(len(v), xi), v, c='k', s=14, zorder=3)

# architecture effect under matched Adam, computed from the harvested means
m_mlp, m_res = adam[0][0], adam[1][0]
if np.isfinite(m_mlp) and np.isfinite(m_res) and m_mlp > 0:
    eff = 100 * (m_res - m_mlp) / m_mlp
    ax.annotate('', xy=(x[1]-w/2, m_res), xytext=(x[0]-w/2, m_mlp),
                arrowprops=dict(arrowstyle='<->', color='#4C72B0', lw=1.2))
    ax.text(0.5, (m_mlp+m_res)/2, f'matched Adam:\n+{eff:.0f}%',
            color='#4C72B0', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel(r'Feature-norm threshold  $f_n^\star$'); ax.set_xlabel('Architecture (MNIST)')
ax.legend(frameon=False, loc='upper left'); ax.spines[['top','right']].set_visible(False)
ax.set_title('Optimiser de-confound: architecture effect under matched Adam', fontsize=9.5)
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/nc_outputs/fig_deconfound.pdf', bbox_inches='tight')
fig.savefig('/content/drive/MyDrive/nc_outputs/fig_deconfound.png', dpi=300, bbox_inches='tight')
plt.show()